# T2 — Spam model fine-tuning and evaluation

This notebook is self-contained: install dependencies, download the dataset, create a stratified train/test split, prepare leakage-safe pipelines, and optionally run Optuna tuning followed by one held-out evaluation. The test set is excluded from tuning. Run cells in order. The final cell starts the full configured optimization and evaluation.

## 1. Dependencies and dataset download

The dataset is downloaded from Kaggle using `kagglehub`. Kaggle access may require account credentials or accepting the dataset terms.

In [ ]:
%pip install -q kagglehub optuna==4.9.0 scikit-learn==1.9.0 xgboost==3.4.1 pandas==3.0.5 numpy==2.5.2 threadpoolctl matplotlib


In [ ]:
from pathlib import Path
import kagglehub
import pandas as pd

DATA_DIR = Path("data")
DATA_DIR.mkdir(parents=True, exist_ok=True)
download_path = Path(kagglehub.dataset_download(
    "balaka18/email-spam-classification-dataset-csv", output_dir=str(DATA_DIR)
))
csv_files = sorted(download_path.rglob("*.csv")) if download_path.is_dir() else [download_path]
if not csv_files:
    raise FileNotFoundError(f"No CSV file found in downloaded dataset: {download_path}")
data_path = next((p for p in csv_files if p.name.lower() == "emails.csv"), csv_files[0])
df = pd.read_csv(data_path)
print(f"Loaded {data_path} with shape {df.shape}")


## 2. Target, features, and stratified holdout

`Prediction=1` denotes spam. The identifier is removed. The 20% test set is held aside and must not influence model or hyperparameter selection.

In [ ]:
from sklearn.model_selection import train_test_split

TARGET = "Prediction"
IDENTIFIER = "Email No."
if TARGET not in df.columns:
    raise KeyError(f"Target column {TARGET!r} is missing")
X = df.drop(columns=[c for c in (IDENTIFIER, TARGET) if c in df.columns])
y = df[TARGET].astype(int)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)
print(f"Train: {X_train.shape}; test: {X_test.shape}; training classes: {y_train.value_counts().to_dict()}")


## 3. Candidate choice and validation design

The candidates are Neural Network, Decision Tree, and XGBoost, following the T1 report and the group's request to add a boosted tree. Spam precision is the primary selection metric; accuracy, F1, and recall are also recorded. All models share stratified five-fold CV (seed 42). Pipelines fit TF-IDF inside each fold. The holdout is used only in the final evaluation.

# T2 - In-depth evaluation

## T2.1 Candidate selection and justification

We select **Neural Network, Decision Tree, and XGBoost** for T2. Section 7 of [the T1 report](../reports/T1_Report.pdf) selected Neural Network and Decision Tree to investigate how hyperparameter optimization affects a competitive model and a weaker initial model. XGBoost is added to extend the comparison to a boosted tree ensemble. This explicitly differs from selecting the three highest-ranked algorithms in the T1 notebook.

The following evidence comes exclusively from the five-fold stratified CV on the training portion, as reported in section 5.1 of the T1 report and `results/cross_validation/text/cross_validation_summary_and_ranking.txt`. Entries are **mean +/- sample standard deviation across folds**; they are descriptive statistics, not confidence intervals or proof of significant differences.

| Candidate | Accuracy | F1-score | Precision (spam) | Recall (spam) |
| --- | --- | --- | --- | --- |
| Neural Network | 0.9845 +/- 0.0047 | 0.9735 +/- 0.0081 | 0.9664 +/- 0.0098 | 0.9808 +/- 0.0113 |
| Decision Tree | 0.9081 +/- 0.0056 | 0.8402 +/- 0.0102 | 0.8480 +/- 0.0074 | 0.8325 +/- 0.0130 |
| XGBoost | 0.9662 +/- 0.0071 | 0.9425 +/- 0.0119 | 0.9289 +/- 0.0138 | 0.9567 +/- 0.0137 |

- **Neural Network:** competitive performance across all four metrics even before tuning, including high spam recall. Its low fold variability supports further investigation but does not guarantee generalization. Iterative optimization of a high-dimensional network is expected to make an extensive search relatively costly.
- **Decision Tree:** substantially lower initial performance on every metric. It is retained as the group's explicit investigation of whether controlling model complexity improves a weak baseline, not as a top-ranked performer. Its rule-based structure also offers a different inductive bias from the network and a naturally interpretable alternative. A single tree is generally cheaper to investigate than a network or many boosted trees, but no runtime was measured here.
- **XGBoost:** better initial metrics than the single tree, though below the network. It allows a direct comparison between one tree and a sequential ensemble of trees and tests whether optimization can narrow the gap to the strongest T1 models. Training multiple boosting rounds adds computational cost; exact relative runtimes remain to be measured.

The diversity is between a neural approach and two related tree approaches; the two tree models are not independent model families. Linear SVM was the strongest T1 baseline (accuracy 0.9867 +/- 0.0039, precision 0.9736 +/- 0.0079), and Logistic Regression also outperformed both tree approaches. Their exclusion follows the group's investigative choice, not evidence that the selected models are superior. Tuning is not guaranteed to reverse the T1 ranking. No held-out test result is used to justify this selection.


### Primary metric and continuity with T1

**Precision for spam (class 1)** is the primary T2 selection metric, following section 3 of the submitted T1 report: false positives send normal email to spam and can hide legitimate messages. Precision measures how many messages predicted as spam are actually spam; it is not the false-positive rate and must be interpreted together with recall and the other metrics.

The existing T1 notebook ranks by accuracy, whereas the submitted report prioritizes precision. T2 explicitly follows the report's precision objective; the historical T1 code and ranking are preserved. Accuracy, F1-score, and recall remain mandatory complementary metrics, especially because spam represents approximately 29% of the dataset. A high precision with very low recall would not be sufficient evidence of a useful model. No decision-threshold optimization is introduced.


## T2.2 Validation protocols prepared before tuning

Both protocols reuse the original count matrix `X_train` and labels `y_train` from the existing stratified 80/20 split (`random_state=42`). There is no new holdout split. The original 20% test holdout remains excluded from all T2 preparation and optimization. Its historical T1 evaluation is descriptive only and must not guide T2 settings.

**Execution prerequisites:** load the dataset and establish the existing `X_train`/`y_train` split before running these T2 cells. The T1 spot-checking and test-evaluation loops are not prerequisites and must not be rerun merely to prepare T2. The cells below only construct partitions and scoring objects; they do not fit preprocessing or models.

### Conventional stratified cross-validation

`t2_standard_splits` contains the same five shuffled, stratified folds as the T1 CV, using seed 42. Each future search will receive the raw `X_train`, `y_train`, and this shared list as its validation splits. Configuration selection will maximize mean validation precision while recording all four metrics. The best search CV score is a selection score and can be optimistic after trying many configurations; it is not an independent estimate of the tuning procedure's performance.

### Nested stratified cross-validation

`t2_nested_splits` contains five outer folds and three inner folds per outer training subset. Outer folds use the same five-fold partitioning as conventional CV. Each inner split is generated using only that outer fold's training data, with seed 42. All three candidate models will reuse these exact partitions.

For future execution, slice `X_train` and `y_train` with a fold's `train_indices` using `.iloc`, then pass its `inner_splits` to a fresh search on that subset. Inner indices are **positions relative to the outer training subset**, not positions in the full training data or DataFrame index labels. After inner selection, refit the chosen pipeline on the entire outer training subset and evaluate only on the corresponding `validation_indices` from the outer fold. Outer validation data must not influence the inner search, preprocessing, or thresholds. Outer scores will summarize the tuning procedure's generalization and variability, rather than choose the inner configuration.

Five outer folds preserve continuity with T1; three inner folds limit the cost, especially for the network and boosted ensemble. For K configurations, conventional CV needs 5K candidate fits, while nested CV needs 15K candidate fits plus five outer-training refits. Additional final searches/refits are not included in those counts. Nested CV offers separation of selection and evaluation at greater computational cost; both protocols are prepared now, without using test performance to choose between them.

### Scoring and future execution boundary

`t2_scoring` defines accuracy, F1-score, precision, and recall. Spam (`1`) is positive, and undefined precision/recall/F1 receive zero. `t2_primary_metric = "precision"` declares the future optimization objective; the tuning method, search spaces, budget, and any supported random seeds must be fixed before optimization begins. No threshold tuning is introduced. Only after optimization is complete may the selected pipelines be fitted on all training data and compared on the held-out test set. No such execution occurs here.


In [8]:
from sklearn.model_selection import StratifiedKFold as t2_StratifiedKFold
from sklearn.metrics import (
    make_scorer as t2_make_scorer,
    accuracy_score as t2_accuracy_score,
    f1_score as t2_f1_score,
    precision_score as t2_precision_score,
    recall_score as t2_recall_score,
)

t2_random_state = 42
t2_primary_metric = "precision"
t2_scoring = {
    "accuracy": t2_make_scorer(t2_accuracy_score),
    "f1_score": t2_make_scorer(t2_f1_score, pos_label=1, zero_division=0),
    "precision": t2_make_scorer(t2_precision_score, pos_label=1, zero_division=0),
    "recall": t2_make_scorer(t2_recall_score, pos_label=1, zero_division=0),
}

t2_standard_cv = t2_StratifiedKFold(
    n_splits=5, shuffle=True, random_state=t2_random_state
)
t2_standard_splits = list(t2_standard_cv.split(X_train, y_train))

t2_outer_cv = t2_StratifiedKFold(
    n_splits=5, shuffle=True, random_state=t2_random_state
)
t2_inner_cv = t2_StratifiedKFold(
    n_splits=3, shuffle=True, random_state=t2_random_state
)
t2_nested_splits = [
    {
        "fold": t2_fold,
        "train_indices": t2_train_indices,
        "validation_indices": t2_validation_indices,
        "inner_splits": list(t2_inner_cv.split(
            X_train.iloc[t2_train_indices], y_train.iloc[t2_train_indices]
        )),
    }
    for t2_fold, (t2_train_indices, t2_validation_indices) in enumerate(
        t2_outer_cv.split(X_train, y_train), start=1
    )
]


## T2.3 Leakage-safe pipelines (unfitted)

`t2_pipelines` holds independent pipeline templates for the three selected algorithms. Every template contains a fresh `TfidfTransformer` followed by a fresh estimator. The TF-IDF configuration and initial estimator settings match T1; these are starting configurations, not optimized hyperparameters. Estimator seeds are fixed at 42, and XGBoost retains `n_jobs=1`. No template reuses a fitted T1 transformer or model.

The CSV already contains word counts, so `TfidfTransformer` is appropriate; tokenization with `TfidfVectorizer` is unnecessary. Future searches must receive **raw count rows** from `X_train` and the corresponding `y_train`, together with a fresh clone of the entire pipeline. They must not receive `X_train_tfidf`, `X_test_tfidf`, or the pre-transformed `fold_data` from T1. In a scikit-learn search, parameter names for the estimator will use the `model__` prefix. No parameter space or search method is defined in this preparation.

During conventional CV, each future search fit will learn IDF only from that fold's training rows. During nested CV, IDF will be learned only from each inner training fold; after inner selection, refitting on the outer training subset will learn a new IDF without access to the outer validation subset. The fitted pipeline will apply its learned transformation to validation data without refitting. Any future learned feature selection, balancing, or imputation must follow the same fold-local rule and be included in an appropriate pipeline.

The holdout must not participate in fitting preprocessing, choosing features, hyperparameters, or thresholds. Once all search decisions are frozen and tuning is complete, the selected full pipelines can be refitted on all `X_train`/`y_train` and applied once for the definitive held-out comparison. The existing T1 test results do not guide those decisions.

**Current boundary:** the following cell only constructs unfitted pipeline templates. It does not train, transform data, predict, calculate evaluation results, or write experiment outputs. Training and optimization remain deferred until the hyperparameter-tuning method is chosen.


In [9]:
from sklearn.feature_extraction.text import TfidfTransformer as t2_TfidfTransformer
from sklearn.pipeline import Pipeline as t2_Pipeline
from sklearn.neural_network import MLPClassifier as t2_MLPClassifier
from sklearn.tree import DecisionTreeClassifier as t2_DecisionTreeClassifier
from xgboost import XGBClassifier as t2_XGBClassifier

t2_model_templates = {
    "Neural Network": t2_MLPClassifier(
        hidden_layer_sizes=(100,),
        activation="relu",
        solver="adam",
        max_iter=500,
        random_state=t2_random_state,
    ),
    "Decision Tree": t2_DecisionTreeClassifier(random_state=t2_random_state),
    "XGBoost": t2_XGBClassifier(
        objective="binary:logistic",
        tree_method="hist",
        random_state=t2_random_state,
        n_jobs=1,
    ),
}

t2_pipelines = {
    t2_model_name: t2_Pipeline([
        ("tfidf", t2_TfidfTransformer(
            norm="l2", use_idf=True, smooth_idf=True, sublinear_tf=True
        )),
        ("model", t2_model_template),
    ])
    for t2_model_name, t2_model_template in t2_model_templates.items()
}


## T2.4 Revised Optuna pipeline: conventional CV, five trials (not executed)

The current instruction is **preparation only**: use conventional stratified five-fold CV with **five Optuna trials per model**, and do not execute training, tuning, or held-out evaluation. Neural Network, Decision Tree, and XGBoost remain the candidates, with spam precision as the primary metric and accuracy, F1-score, and recall recorded alongside it.

The original 20-trial conventional-plus-nested execution was stopped at the user's request. Its frozen implementation is preserved in Git commit `8bc91d9`. The historical 20-trial Neural Network conventional study and first nested inner study completed; the second nested inner study stopped after 13 completed trials. One outer evaluation completed. Their existing TXT/PDF evidence is retained as an interrupted experiment, not as results of the revised five-trial protocol. Neither Decision Tree nor XGBoost ran in that experiment, and the holdout was not evaluated.

The functions below prepare **three independent conventional-CV studies**, each with `TPESampler(seed=42, n_startup_trials=2)`: two initial trials (including the T1 baseline), followed by three adaptive proposals. There is no pruning, timeout, or threshold tuning. Trials and fits remain sequential with numerical thread pools limited to one. The future logical budget is **75 CV fits** (3 models x 5 trials x 5 folds), plus three final refits if later authorized. No execution call is made by the new preparation cell. The original five-outer/three-inner partition definitions remain available in T2.2, but nested training is deferred.

| Model | Prepared search space (unchanged) |
| --- | --- |
| Neural Network | hidden layers: (50,), (100,), (50, 50), (100, 50); activation: relu/tanh; alpha: 1e-6 to 1e-1 (log); learning_rate_init: 1e-4 to 1e-2 (log); batch_size: auto/64/128/256. Adam, max_iter=500, early_stopping=False remain fixed. |
| Decision Tree | criterion: gini/entropy; max_depth: None/5/10/20/40; min_samples_split: 2 to 30; min_samples_leaf: 1 to 20; max_features: None/sqrt/log2; ccp_alpha: 0 to 0.02. |
| XGBoost | n_estimators: 50 to 300, step 50; max_depth: 2 to 10; learning_rate: 0.01 to 0.3 (log); min_child_weight: 1 to 10; subsample: 0.6 to 1; colsample_bytree: 0.5 to 1; reg_lambda: 1e-3 to 10 (log); reg_alpha: 0 to 1; gamma: 0 to 5. binary:logistic, hist, n_jobs=1 remain fixed. |

These spaces investigate network capacity/regularization, tree complexity/pruning, and boosting capacity/sampling/regularization. Five trials provide limited exploration, not evidence of a global optimum. All other initial settings remain fixed; effective XGBoost baseline defaults are explicit. Exact precision ties select the earliest completed trial.

Every future fit must clone the complete TF-IDF/estimator pipeline and receive original count rows from `X_train`, never the T1 pre-transformed matrices. The same five folds are shared by all candidates. Future reports will retain fold metrics, sample standard deviations (`ddof=1`), parameters, timings, warnings, failures, and versions. New five-trial evidence is isolated under `results/hyperparameter_optimization/text/standard_5_trials/` and the corresponding `figures/standard_5_trials/`, preserving the earlier run.

**Do not run the optimization function until training is explicitly requested.** The prepared entry point is `t2_run_optimization(X_train, y_train)`. Do not rerun T1 experiments to prepare this section. No results for the five-trial design exist yet.

References: [Optuna reproducibility](https://optuna.readthedocs.io/en/stable/faq.html), [nested versus conventional CV](https://scikit-learn.org/stable/auto_examples/model_selection/plot_nested_cross_validation_iris.html).


In [10]:
# Optional environment setup, only when dependency installation is authorized/needed.
%pip install -q optuna==4.9.0 scikit-learn==1.9.0 xgboost==3.4.1 pandas==3.0.5 threadpoolctl matplotlib


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import json as t2_json
import platform as t2_platform
import time as t2_time
import warnings as t2_warnings
from pathlib import Path as t2_Path
from importlib.metadata import version as t2_version

import numpy as t2_np
import pandas as t2_pd
import matplotlib.pyplot as t2_plt
import optuna as t2_optuna
from sklearn.base import clone as t2_clone
from threadpoolctl import threadpool_limits as t2_threadpool_limits

t2_trial_budget = 20
t2_startup_trials = 10
t2_current_protocol = f"standard_{t2_trial_budget + t2_startup_trials}_trials"
t2_optuna.logging.set_verbosity(t2_optuna.logging.WARNING)
t2_layers = {"50": (50,), "100": (100,), "50_50": (50, 50), "100_50": (100, 50)}
t2_baselines = {
    "Neural Network": {"layers": "100", "activation": "relu", "alpha": 0.0001,
                       "learning_rate_init": 0.001, "batch_size": "auto"},
    "Decision Tree": {"criterion": "gini", "max_depth": None, "min_samples_split": 2,
                      "min_samples_leaf": 1, "max_features": None, "ccp_alpha": 0.0},
    "XGBoost": {"n_estimators": 100, "max_depth": 6, "learning_rate": 0.3,
                "min_child_weight": 1, "subsample": 1.0, "colsample_bytree": 1.0,
                "reg_lambda": 1.0, "reg_alpha": 0.0, "gamma": 0.0},
}
t2_versions = {"python": t2_platform.python_version(), **{
    t2_package: t2_version(t2_package)
    for t2_package in ("optuna", "scikit-learn", "xgboost", "pandas", "numpy", "threadpoolctl", "matplotlib")
}}
# Resolve output paths independently of whether the notebook runs from repo root or src/.
t2_source_dir = t2_Path.cwd() / "src" if (t2_Path.cwd() / "src/spam-detection.ipynb").exists() else t2_Path.cwd()
t2_run_root = t2_source_dir / "results" / "hyperparameter_optimization"
t2_text_root = t2_run_root / "text"
t2_figure_root = t2_run_root / "figures"


def t2_write_report(t2_destination, t2_content):
    t2_destination.parent.mkdir(parents=True, exist_ok=True)
    t2_destination.write_text(t2_content.rstrip() + "\n", encoding="utf-8")


def t2_suggest_parameters(t2_trial, t2_model_name):
    if t2_model_name == "Neural Network":
        return {
            "hidden_layer_sizes": t2_layers[t2_trial.suggest_categorical("layers", list(t2_layers))],
            "activation": t2_trial.suggest_categorical("activation", ["relu", "tanh"]),
            "alpha": t2_trial.suggest_float("alpha", 1e-6, 1e-1, log=True),
            "learning_rate_init": t2_trial.suggest_float("learning_rate_init", 1e-4, 1e-2, log=True),
            "batch_size": t2_trial.suggest_categorical("batch_size", ["auto", 64, 128, 256]),
        }
    if t2_model_name == "Decision Tree":
        return {
            "criterion": t2_trial.suggest_categorical("criterion", ["gini", "entropy"]),
            "max_depth": t2_trial.suggest_categorical("max_depth", [None, 5, 10, 20, 40]),
            "min_samples_split": t2_trial.suggest_int("min_samples_split", 2, 30),
            "min_samples_leaf": t2_trial.suggest_int("min_samples_leaf", 1, 20),
            "max_features": t2_trial.suggest_categorical("max_features", [None, "sqrt", "log2"]),
            "ccp_alpha": t2_trial.suggest_float("ccp_alpha", 0.0, 0.02),
        }
    if t2_model_name == "XGBoost":
        return {
            "n_estimators": t2_trial.suggest_int("n_estimators", 50, 300, step=50),
            "max_depth": t2_trial.suggest_int("max_depth", 2, 10),
            "learning_rate": t2_trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            "min_child_weight": t2_trial.suggest_int("min_child_weight", 1, 10),
            "subsample": t2_trial.suggest_float("subsample", 0.6, 1.0),
            "colsample_bytree": t2_trial.suggest_float("colsample_bytree", 0.5, 1.0),
            "reg_lambda": t2_trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
            "reg_alpha": t2_trial.suggest_float("reg_alpha", 0.0, 1.0),
            "gamma": t2_trial.suggest_float("gamma", 0.0, 5.0),
        }
    raise ValueError(t2_model_name)


def t2_configured_pipeline(t2_model_name, t2_params):
    t2_params = dict(t2_params)
    if "hidden_layer_sizes" in t2_params:
        t2_params["hidden_layer_sizes"] = tuple(t2_params["hidden_layer_sizes"])
    return t2_clone(t2_pipelines[t2_model_name]).set_params(**{
        "model__" + t2_key: t2_value for t2_key, t2_value in t2_params.items()
    })


def t2_metrics(t2_labels, t2_predictions):
    return {
        "accuracy": float(t2_accuracy_score(t2_labels, t2_predictions)),
        "f1_score": float(t2_f1_score(t2_labels, t2_predictions, pos_label=1, zero_division=0)),
        "precision": float(t2_precision_score(t2_labels, t2_predictions, pos_label=1, zero_division=0)),
        "recall": float(t2_recall_score(t2_labels, t2_predictions, pos_label=1, zero_division=0)),
    }


def t2_summarize_metrics(t2_rows):
    return {t2_metric: {
        "mean": float(t2_np.mean([t2_row[t2_metric] for t2_row in t2_rows])),
        "std": float(t2_np.std([t2_row[t2_metric] for t2_row in t2_rows], ddof=1)),
    } for t2_metric in t2_scoring}


def t2_fit_predict(t2_model_name, t2_params, t2_fit_X, t2_fit_y, t2_eval_X, t2_eval_y):
    t2_start = t2_time.perf_counter()
    t2_pipeline = t2_configured_pipeline(t2_model_name, t2_params)
    with t2_threadpool_limits(limits=1), t2_warnings.catch_warnings(record=True) as t2_caught:
        t2_warnings.simplefilter("always")
        t2_pipeline.fit(t2_fit_X, t2_fit_y)
        t2_predictions = t2_pipeline.predict(t2_eval_X)
    return t2_pipeline, {
        **t2_metrics(t2_eval_y, t2_predictions),
        "duration_seconds": t2_time.perf_counter() - t2_start,
        "warnings": [{"category": t2_warning.category.__name__, "message": str(t2_warning.message)}
                     for t2_warning in t2_caught],
    }


def t2_study_report(t2_study, t2_model_name, t2_protocol, t2_elapsed):
    t2_trials = [{
        "number": t2_trial.number, "state": t2_trial.state.name,
        "value": t2_trial.value, "parameters": t2_trial.params,
        "duration_seconds": t2_trial.duration.total_seconds() if t2_trial.duration else None,
        **t2_trial.user_attrs,
    } for t2_trial in t2_study.trials]
    t2_complete = [t2_trial for t2_trial in t2_trials if t2_trial["state"] == "COMPLETE"]
    t2_best = max(t2_complete, key=lambda t2_trial: (t2_trial["value"], -t2_trial["number"])) if t2_complete else None
    return {
        "model": t2_model_name, "protocol": t2_protocol, "seed": 42,
        "budget": t2_trial_budget, "sampler": "TPESampler(n_startup_trials=2, seed=42)",
        "pruner": "NopPruner", "primary_metric": t2_primary_metric,
        "elapsed_seconds": t2_elapsed, "versions": t2_versions,
        "complete": len(t2_complete) == t2_trial_budget and len(t2_trials) == t2_trial_budget,
        "best_trial": t2_best, "trials": t2_trials,
    }


def t2_run_study(t2_model_name, t2_protocol, t2_X, t2_y, t2_splits):
    t2_slug = t2_model_name.lower().replace(" ", "_")
    t2_report_path = t2_text_root / t2_protocol / (t2_slug + ".txt")
    if t2_report_path.exists():
        raise FileExistsError(f"Preserve existing evidence: {t2_report_path}")
    t2_study = t2_optuna.create_study(
        direction="maximize", sampler=t2_optuna.samplers.TPESampler(seed=42, n_startup_trials=t2_startup_trials),
        pruner=t2_optuna.pruners.NopPruner(),
    )
    t2_study.enqueue_trial(t2_baselines[t2_model_name])
    t2_start = t2_time.perf_counter()

    def t2_objective(t2_trial):
        t2_params = t2_suggest_parameters(t2_trial, t2_model_name)
        t2_trial.set_user_attr("model_parameters", t2_params)
        t2_rows = []
        try:
            for t2_fold, (t2_train, t2_valid) in enumerate(t2_splits, 1):
                _, t2_row = t2_fit_predict(
                    t2_model_name, t2_params, t2_X.iloc[t2_train], t2_y.iloc[t2_train],
                    t2_X.iloc[t2_valid], t2_y.iloc[t2_valid],
                )
                t2_rows.append({"fold": t2_fold, **t2_row})
                t2_trial.set_user_attr("folds", t2_rows)
            t2_summary = t2_summarize_metrics(t2_rows)
            t2_trial.set_user_attr("metrics", t2_summary)
            return t2_summary[t2_primary_metric]["mean"]
        except Exception as t2_error:
            t2_trial.set_user_attr("failure", repr(t2_error))
            raise

    def t2_callback(t2_study, t2_trial):
        t2_report = t2_study_report(t2_study, t2_model_name, t2_protocol, t2_time.perf_counter() - t2_start)
        t2_write_report(t2_report_path, t2_json.dumps(t2_report, indent=2, allow_nan=False))
        print(f"{t2_model_name} {t2_protocol}: trial {t2_trial.number + 1}/{t2_trial_budget} "
              f"precision={t2_trial.value} elapsed={t2_report['elapsed_seconds']:.1f}s", flush=True)

    try:
        t2_study.optimize(t2_objective, n_trials=t2_trial_budget, n_jobs=1, callbacks=[t2_callback])
    finally:
        t2_report = t2_study_report(t2_study, t2_model_name, t2_protocol, t2_time.perf_counter() - t2_start)
        t2_write_report(t2_report_path, t2_json.dumps(t2_report, indent=2, allow_nan=False))
    if not t2_report["complete"]:
        raise RuntimeError("Incomplete study; final evaluation is prohibited")
    t2_figure_path = t2_figure_root / t2_protocol / (t2_slug + "_optimization_history.pdf")
    t2_figure_path.parent.mkdir(parents=True, exist_ok=True)
    t2_values = [t2_trial["value"] for t2_trial in t2_report["trials"]]
    t2_fig, t2_ax = t2_plt.subplots(figsize=(8, 4))
    t2_ax.plot(range(1, t2_trial_budget + 1), t2_values, "o-", label="Trial mean precision")
    t2_ax.plot(range(1, t2_trial_budget + 1), t2_np.maximum.accumulate(t2_values), label="Best so far")
    t2_ax.set(xlabel="Trial", ylabel="Spam precision", title=f"{t2_model_name}: {t2_protocol}")
    t2_ax.legend()
    t2_fig.tight_layout()
    t2_fig.savefig(t2_figure_path, format="pdf", bbox_inches="tight")
    t2_plt.close(t2_fig)
    return t2_report


def t2_run_optimization(t2_X, t2_y):
    """Execute only after explicit training authorization; preparation never calls this."""
    for t2_name in t2_pipelines:
        t2_path = t2_text_root / t2_current_protocol / (t2_name.lower().replace(" ", "_") + ".txt")
        if t2_path.exists():
            raise FileExistsError(f"Existing evidence: {t2_path}; select a new run root explicitly")
    t2_start = t2_time.perf_counter()
    t2_reports = {
        t2_name: t2_run_study(t2_name, t2_current_protocol, t2_X, t2_y, t2_standard_splits)
        for t2_name in t2_pipelines
    }
    t2_summary = {t2_name: {
        "standard_selected_cv": t2_report["best_trial"]["metrics"],
        "standard_parameters": t2_report["best_trial"]["model_parameters"],
    } for t2_name, t2_report in t2_reports.items()}
    t2_result = {
        "complete": True, "protocol": t2_current_protocol,
        "studies": len(t2_reports), "trials": t2_trial_budget * len(t2_reports),
        "cv_fits": t2_trial_budget * len(t2_standard_splits) * len(t2_reports),
        "outer_refits": 0, "elapsed_seconds": t2_time.perf_counter() - t2_start,
        "versions": t2_versions, "models": t2_summary,
    }
    t2_write_report(t2_text_root / t2_current_protocol / "optimization_summary.txt",
                    t2_json.dumps(t2_result, indent=2))
    return t2_result


In [12]:
# Preparation only: this dictionary describes the future searches; it does not run them.
t2_prepared_searches = {
    t2_name: {
        "pipeline": t2_clone(t2_template),
        "cv_splits": t2_standard_splits,
        "scoring": t2_scoring,
        "primary_metric": t2_primary_metric,
        "n_trials": t2_trial_budget,
        "n_startup_trials": t2_startup_trials,
        "seed": t2_random_state,
        "n_jobs": 1,
    }
    for t2_name, t2_template in t2_pipelines.items()
}
# No call to t2_run_optimization, fit, predict, or held-out evaluation.


## T2.5 Held-out evaluation function (deferred, not executed)

The definition below is retained for future use only. It is not called. Training and final evaluation require a new explicit request from the user.

After all three revised five-trial conventional searches finish, their selected configurations can be refitted on all training data and evaluated once on the holdout. The function checks the revised protocol, three completed studies, and 15 total trials, and refuses to overwrite existing final evidence. Reports compare test metrics with selected conventional-CV means; nested results from the interrupted earlier run must not be combined with this revised protocol.

The holdout remains unevaluated in T2. No final-model ranking or test conclusions are available. Confusion matrices, curves, error analysis, model interpretation, Model Card, and presentation remain future work.


In [13]:
def t2_evaluate_finalists(t2_optimization, t2_train_X, t2_train_y, t2_test_X, t2_test_y):
    t2_expected_studies = len(t2_pipelines)
    t2_expected_trials = t2_trial_budget * t2_expected_studies
    if (not t2_optimization.get("complete")
            or t2_optimization.get("protocol") != t2_current_protocol
            or t2_optimization.get("studies") != t2_expected_studies
            or t2_optimization.get("trials") != t2_expected_trials
            or set(t2_optimization.get("models", {})) != set(t2_pipelines)):
        raise RuntimeError(
            f"All {t2_expected_studies} conventional studies with "
            f"{t2_trial_budget} trials each ({t2_expected_trials} total) "
            f"must finish under protocol {t2_current_protocol} before final evaluation"
        )
    t2_final_dir = t2_source_dir / "results" / "final_comparison" / "text"
    if t2_final_dir.exists() and any(t2_final_dir.iterdir()):
        raise FileExistsError("Final evidence already exists; do not repeat held-out evaluation")
    t2_final_dir.mkdir(parents=True, exist_ok=True)
    t2_final_models, t2_final_rows = {}, []
    for t2_name, t2_selection in t2_optimization["models"].items():
        t2_start = t2_time.perf_counter()
        t2_model = t2_configured_pipeline(t2_name, t2_selection["standard_parameters"])
        with t2_threadpool_limits(limits=1), t2_warnings.catch_warnings(record=True) as t2_caught:
            t2_warnings.simplefilter("always")
            t2_model.fit(t2_train_X, t2_train_y)
            t2_pred = t2_model.predict(t2_test_X)
        t2_scores = t2_metrics(t2_test_y, t2_pred)
        t2_row = {"model": t2_name, **t2_scores,
                  "duration_seconds": t2_time.perf_counter() - t2_start,
                  "warnings": [str(t2_warning.message) for t2_warning in t2_caught],
                  "parameters": t2_selection["standard_parameters"],
                  "test_minus_standard_cv": {t2_k: t2_scores[t2_k] - t2_selection["standard_selected_cv"][t2_k]["mean"] for t2_k in t2_scores}}
        t2_final_rows.append(t2_row)
        t2_final_models[t2_name] = t2_model
        t2_slug = t2_name.lower().replace(" ", "_")
        t2_write_report(t2_final_dir / (t2_slug + "_predictions.txt"),
                        t2_pd.DataFrame({"y_true": t2_np.asarray(t2_test_y), "y_pred": t2_pred}).to_string(index=False))
        t2_write_report(t2_final_dir / (t2_slug + "_metrics.txt"), t2_json.dumps(t2_row, indent=2))
        print(t2_name, t2_scores, flush=True)
    t2_write_report(t2_final_dir / "held_out_comparison.txt", t2_json.dumps({"versions": t2_versions, "models": t2_final_rows}, indent=2))
    return t2_final_models, t2_final_rows


# Function definition only. No refit or test evaluation is invoked.


## 4. Run tuning and final evaluation

Run this cell only when ready to start the Optuna searches. It performs the configured trials for all three models, then refits the selected configurations on the full training partition and evaluates each once on the held-out test set. This may take substantial time.

In [ ]:
t2_results = t2_run_optimization(X_train, y_train)
t2_final_models, t2_final_rows = t2_evaluate_finalists(
    t2_optimization=t2_results,
    t2_train_X=X_train, t2_train_y=y_train,
    t2_test_X=X_test, t2_test_y=y_test,
)
